# Differentiation & Derivatives

In [42]:
import sys
assert sys.version_info >= (3, 7)

import numpy as np
import autograd.numpy as anp 
from autograd import grad
import sympy
from sympy import symbols, diff, sin

## Symbolic differentiation

### with `Sympy` (symbolic solution)

In [4]:
# Code snippet in the cell provided by Kelsey (2023)
x = symbols('x')
f = 5*x**3 + 4*x**2 + 15
f_prime = diff(f, x)
print(f_prime)

x = symbols('x')
g = sin(x) + 0.5
g_prime = diff(g, x)
print(g_prime)

x, y = symbols('x y')    
f = x**2 + 3*y**2 + 2*x*y
f_partial_x = diff(f, x)
f_partial_y = diff(f, y)
print(f_partial_x)
print(f_partial_y)

15*x**2 + 8*x
cos(x)
2*x + 2*y
2*x + 6*y


### with `autograd` (numerical solution)

In [46]:
def f(x):
    return 5*x**3 + 4*x**2 + 15

f_prime = grad(f)
print(f_prime(2.0))

def g(x):
    return anp.sin(x) + 0.5

g_prime = grad(g)
print(g_prime(1.0))

def h(vars):
    x, y = vars[0], vars[1]
    return x**2 + 3*y**2 + 2*x*y

h_grad = grad(h)   # returns gradient vector (as an array)

v = anp.array([2.0, 3.0])
gx, gy = h_grad(v) # Returns the partial derivatives wrt x and y

print(gx)  # ∂h/∂x
print(gy)  # ∂h/∂y

76.0
0.5403023058681398
10.0
22.0


## Numeric differentiation

In [48]:
# Code snippet in the cell provided by Kelsey (2023)
def f(x):
    return np.sin(x)

def central_difference(f, x, h):
    """
    Approximate the first derivative of a scalar function using the
    central difference method.

    Args:
        f (callable):
            Scalar function f(x) whose derivative is to be approximated.
        x (float):
            Point at which the derivative is evaluated.
        h (float):
            Step size used for the finite difference.

    Returns:
        float:
            Numerical approximation of f'(x) using
            (f(x + h) - f(x - h)) / (2h).
    
    """
    
    return (f(x + h) - f(x - h)) / (2 * h)

In [49]:
# Code snippet in the cell provided by Kelsey (2023)
x = 0.5
h = 0.01
cos_x = np.cos(0.5)

f_prime = central_difference(f, x, h)
print(f_prime)
print(cos_x)   # compare to exact value

0.8775679355874727
0.8775825618903728


In [50]:
def five_point_derivative(f, x, h):
    """
    Approximate the first derivative of a scalar function using a
    5-point central finite difference stencil.

    This method uses multiple steps on each side of x and achieves
    fourth-order accuracy. The approximation is given by:

        f'(x) ≈ [ -f(x + 2h) + 8f(x + h)
                  -8f(x - h) + f(x - 2h) ] / (12h)

    Args:
        f (callable):
            Scalar function f(x) whose derivative is to be approximated.
        x (float):
            Point at which the derivative is evaluated.
        h (float):
            Step size used for the finite difference.

    Returns:
        float:
            Numerical approximation of f'(x) using a 5-point stencil.
    
    """
    
    return (-f(x + 2*h) + 8*f(x + h) - 8*f(x - h) + f(x - 2*h)) / (12*h)

f_prime = five_point_derivative(f, x, h)
print(f_prime)
print(cos_x)

0.8775825615978473
0.8775825618903728


Given the same stepsize `h`, the 5-point central finite difference stencil is much more precise than the central difference method, specifically it approximates to the ground truth value up to 9 floating point digits, compare to the 4 of the central difference. This is because while the central difference method uses only 2 points (one step on each side), the 5-point stencil uses 4 points (two steps on each side), hence it is more precise. This could be even increases, for example to a 7-point stencil, which uses 6 points.

In [34]:
def central_diff_multiple_h(f, x, h_list):
    f_prime_list = [] # Initialise the results list
    cos_x = np.cos(x)
    print(f"Ground truth: cos({x}) = {cos_x}")
    for h in h_list:
        f_prime = central_difference(f, x, h)
        f_prime_list.append(f_prime)
        close = np.allclose(f_prime, cos_x, rtol=1e-06, atol=1e-08, equal_nan=False)
        print(f"f_prime for h={h}: {f_prime}; close: {close}")
    return f_prime_list

In [35]:
h_list = np.linspace(0.01, 1e-20, num=50)

In [36]:
f_prime_list = central_diff_multiple_h(f, x, h_list)

Ground truth: cos(0.5) = 0.8775825618903728
f_prime for h=0.01: 0.8775679355874727; close: False
f_prime for h=0.009795918367346938: 0.877568526484839; close: False
f_prime for h=0.009591836734693878: 0.8775691051989837; close: False
f_prime for h=0.009387755102040816: 0.8775696717298956; close: False
f_prime for h=0.009183673469387756: 0.8775702260775586; close: False
f_prime for h=0.008979591836734694: 0.8775707682419622; close: False
f_prime for h=0.008775510204081634: 0.8775712982230877; close: False
f_prime for h=0.008571428571428572: 0.877571816020925; close: False
f_prime for h=0.00836734693877551: 0.8775723216354626; close: False
f_prime for h=0.00816326530612245: 0.8775728150666846; close: False
f_prime for h=0.007959183673469388: 0.8775732963145806; close: False
f_prime for h=0.007755102040816327: 0.8775737653791316; close: False
f_prime for h=0.007551020408163266: 0.877574222260339; close: False
f_prime for h=0.007346938775510204: 0.8775746669581852; close: False
f_prime for

In [37]:
print(np.cos(0.5))

0.8775825618903728


Decreasing stepsize increases finite difference precision, however, if stepsize becomes too small, it may lead to floating points errors. In fact, in the example, it can be seen that as stepsizes decreases, the `f_prime` becomes close to the ground truth `cos(x)` within at least `rtol=1e-06`, but with `f_prime=0.0` for `h=1e-20`, because the perturbation is `h` is so small that the machine sees it as `0.0`, so `x + h` and `x - h` become the same value, and the subtraction  `f(x + h) - f(x - h) = 0`. In other words, the derivative estimate collapses to zero instead of approximating the true derivative.

REFERENCES
- Kelsey, T. (2023). Topic 3: QR, SVD & Derivatives [Lecture Notebook]. University of St Andrews.